In [1]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import glob
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import os
import pandas as pd
import gc
from pathlib import Path
import netCDF4 as nc
from datetime import datetime
import re

In [2]:
gemlam_dir = "/results/forcing/atmospheric/GEM2.5/gemlam"
operational_dir = "/results/forcing/atmospheric/GEM2.5/operational"
time_fixed_dir = "/ocean/dtaneja/MOAD/analysis-dishika/notebooks/TimeFixed"

gemlam_start = datetime(2007, 1, 3)
gemlam_end = datetime(2014, 9, 30)
ops_start = datetime(2014, 10, 1)
ops_end = datetime(2021, 12, 31)

years = range(2007, 2022)

fixed_2008_filenames = {
    "gemlam_y2008m07d16.nc",
    "gemlam_y2008m07d17.nc",
    "gemlam_y2008m07d18.nc",
    "gemlam_y2008m07d19.nc",
    "gemlam_y2008m07d20.nc",
    "gemlam_y2008m07d21.nc",
    "gemlam_y2008m07d22.nc",
    "gemlam_y2008m07d23.nc",
    "gemlam_y2008m08d10.nc",
    "gemlam_y2008m08d11.nc",
    "gemlam_y2008m08d12.nc",
}

def get_file_date(filepath):
    filename = os.path.basename(filepath)
    match = re.search(r"_y(\d{4})m(\d{2})d(\d{2})", filename)
    if match is None:
        return None
    year, month, day = map(int, match.groups())
    return datetime(year, month, day)

all_gemlam_files = glob.glob(os.path.join(gemlam_dir, "*.nc"))
all_ops_files = glob.glob(os.path.join(operational_dir, "ops_y????m??d??.nc"))

selected_gemlam_files = {}

for filepath in all_gemlam_files:
    file_date = get_file_date(filepath)
    if file_date is not None and gemlam_start <= file_date <= gemlam_end:
        selected_gemlam_files[file_date] = filepath

for filename in fixed_2008_filenames:
    fixed_filepath = os.path.join(time_fixed_dir, filename)
    if not os.path.exists(fixed_filepath):
        raise FileNotFoundError(f"Corrected file not found: {fixed_filepath}")
    fixed_date = get_file_date(fixed_filepath)
    if fixed_date is None:
        raise ValueError(f"Could not extract date from: {fixed_filepath}")
    selected_gemlam_files[fixed_date] = fixed_filepath

selected_ops_files = {}

for filepath in all_ops_files:
    file_date = get_file_date(filepath)
    if file_date is not None and ops_start <= file_date <= ops_end:
        selected_ops_files[file_date] = filepath

all_selected_files = {**selected_gemlam_files, **selected_ops_files}

hrdps_files_by_year = {}

for year in years:
    files = [filepath for file_date, filepath in sorted(all_selected_files.items()) if file_date.year == year]
    if len(files) == 0:
        print(f"{year}: no files found")
        continue
    hrdps_files_by_year[year] = files
    print(year, ":", files[0], "to", files[-1], f"({len(files)} files)")

2007 : /results/forcing/atmospheric/GEM2.5/gemlam/gemlam_y2007m01d03.nc to /results/forcing/atmospheric/GEM2.5/gemlam/gemlam_y2007m12d31.nc (363 files)
2008 : /results/forcing/atmospheric/GEM2.5/gemlam/gemlam_y2008m01d01.nc to /results/forcing/atmospheric/GEM2.5/gemlam/gemlam_y2008m12d31.nc (366 files)
2009 : /results/forcing/atmospheric/GEM2.5/gemlam/gemlam_y2009m01d01.nc to /results/forcing/atmospheric/GEM2.5/gemlam/gemlam_y2009m12d31.nc (365 files)
2010 : /results/forcing/atmospheric/GEM2.5/gemlam/gemlam_y2010m01d01.nc to /results/forcing/atmospheric/GEM2.5/gemlam/gemlam_y2010m12d31.nc (365 files)
2011 : /results/forcing/atmospheric/GEM2.5/gemlam/gemlam_y2011m01d01.nc to /results/forcing/atmospheric/GEM2.5/gemlam/gemlam_y2011m12d31.nc (365 files)
2012 : /results/forcing/atmospheric/GEM2.5/gemlam/gemlam_y2012m01d01.nc to /results/forcing/atmospheric/GEM2.5/gemlam/gemlam_y2012m12d31.nc (366 files)
2013 : /results/forcing/atmospheric/GEM2.5/gemlam/gemlam_y2013m01d01.nc to /results/forc

In [3]:
# Loading datasets

weights_pre_file = "/home/sallen/MEOPAR/grid/weights-gem2.5-gemlam_201702_pre22sep11.nc"
weights_post_file = "/home/sallen/MEOPAR/grid/weights-gem2.5-gemlam_201702_22sep11onward.nc"
mesh_mask_file = "/ocean/dtaneja/MOAD/analysis-dishika/grid/mesh_mask202108.nc"

ds_weights_pre = xr.open_dataset(weights_pre_file).load()
ds_weights_post = xr.open_dataset(weights_post_file).load()

with xr.open_dataset(mesh_mask_file) as ds_mesh:
    nemo_lat = ds_mesh["nav_lat"].load()
    nemo_lon = ds_mesh["nav_lon"].load()

In [4]:
# Converts hourly HRDPS temperature from the 266 × 256 HRDPS grid to the 898 × 398 NEMO grid

transition_time = pd.Timestamp("2011-09-22 00:00:00")

def interpolate_precip_with_weights(precip, ds_weights):
    precip = precip.transpose("time_counter", "y", "x")

    n_time = precip.sizes["time_counter"]
    source_shape = (precip.sizes["y"],precip.sizes["x"])
    n_source_cells = source_shape[0] * source_shape[1]

    source_values = (precip.values.reshape(n_time, n_source_cells).astype(np.float32))
    target_shape = ds_weights["src01"].shape

    interpolated = np.zeros((n_time, target_shape[0], target_shape[1]),dtype=np.float32)

    for n in range(1, 5):
        # The source indexes in the weights file are 1-based
        source_index = (ds_weights[f"src{n:02d}"].values.astype(np.int64)- 1)
        weight = (ds_weights[f"wgt{n:02d}"].values.astype(np.float32))

        interpolated += (source_values[:, source_index]* weight[None, :, :])

    precip_nemo = xr.DataArray(
        interpolated,
        dims=("time_counter", "y", "x"),
        coords={
            "time_counter": precip["time_counter"],
            "nav_lat": (("y", "x"), nemo_lat.values),
            "nav_lon": (("y", "x"), nemo_lon.values),
        },
        name="precip"
    )

    precip_nemo.attrs = precip.attrs.copy()
    precip_nemo.attrs["grid"] = "SalishSeaCast NEMO grid"
    precip_nemo.attrs["interpolation"] = (
        "Four-source weighted HRDPS-to-NEMO interpolation"
    )
    return precip_nemo

In [5]:
# Open file and interpolates

def extract_and_interpolate_hourly_precip(file):
    with xr.open_dataset(file) as ds:
        precip = ds["precip"].sortby("time_counter")
        time_index = precip.get_index("time_counter")

        if time_index.has_duplicates:
            unique_mask = ~time_index.duplicated()
            precip = precip.isel(time_counter=unique_mask)

        precip = precip.load()

    times = pd.to_datetime(precip["time_counter"].values)

    before_transition = times < transition_time
    after_transition = times >= transition_time

    pieces = []

    if before_transition.any():
        precip_pre = precip.isel(time_counter=np.where(before_transition)[0])
        interpolated_pre = interpolate_precip_with_weights(precip_pre,ds_weights_pre)
        pieces.append(interpolated_pre)

    if after_transition.any():
        precip_post = precip.isel(time_counter=np.where(after_transition)[0])
        interpolated_post = interpolate_precip_with_weights(precip_post,ds_weights_post)
        pieces.append(interpolated_post)

    precip_nemo = xr.concat(pieces,dim="time_counter").sortby("time_counter")
    return precip_nemo

In [6]:
sample_pre_file = hrdps_files_by_year[2008][0]
precip_nemo_pre = extract_and_interpolate_hourly_precip(sample_pre_file)

print("Shape:", precip_nemo_pre.shape)
print("Time range:",precip_nemo_pre.time_counter.values[0],"to",precip_nemo_pre.time_counter.values[-1])

Shape: (24, 898, 398)
Time range: 2008-01-01T00:00:00.000000000 to 2008-01-01T23:00:00.000000000


In [7]:
sample_post_file = hrdps_files_by_year[2012][0]
precip_nemo_post = extract_and_interpolate_hourly_precip(sample_post_file)

print("Shape:", precip_nemo_post.shape)
print("Time range:",precip_nemo_post.time_counter.values[0],"to",precip_nemo_post.time_counter.values[-1])

Shape: (24, 898, 398)
Time range: 2012-01-01T00:00:00.000000000 to 2012-01-01T23:00:00.000000000


In [8]:
sample_post_file = hrdps_files_by_year[2015][0]
precip_nemo_post = extract_and_interpolate_hourly_precip(sample_post_file)

print("Shape:", precip_nemo_post.shape)
print("Time range:",precip_nemo_post.time_counter.values[0],"to",precip_nemo_post.time_counter.values[-1])

Shape: (24, 898, 398)
Time range: 2015-01-01T00:00:00.000000000 to 2015-01-01T23:00:00.000000000


In [9]:
sample_post_file = hrdps_files_by_year[2021][-1]
precip_nemo_post = extract_and_interpolate_hourly_precip(sample_post_file)

print("Shape:", precip_nemo_post.shape)
print("Time range:",precip_nemo_post.time_counter.values[0],"to",precip_nemo_post.time_counter.values[-1])

Shape: (24, 898, 398)
Time range: 2021-12-31T00:00:00.000000000 to 2021-12-31T23:00:00.000000000


In [10]:
with xr.open_dataset(mesh_mask_file) as ds_mesh:
    water_mask = (ds_mesh["tmask"].isel(t=0, z=0).load().values.astype(bool))
    nemo_lat_2d = ds_mesh["nav_lat"].load().values
    nemo_lon_2d = ds_mesh["nav_lon"].load().values
nemo_j, nemo_i = np.where(water_mask)
water_flat_indices = np.flatnonzero(water_mask.reshape(-1))
nemo_water_lat = nemo_lat_2d[water_mask]
nemo_water_lon = nemo_lon_2d[water_mask]
n_water = len(water_flat_indices)
print("NEMO grid shape:", water_mask.shape)
print("Number of surface water cells:", n_water)

NEMO grid shape: (898, 398)
Number of surface water cells: 81383


In [11]:
def process_daily_file_to_3h_water(file):
    precip_nemo_hourly = extract_and_interpolate_hourly_precip(file)
    n_time = precip_nemo_hourly.sizes["time_counter"]

    precip_water_values = (precip_nemo_hourly.values.reshape(n_time, -1)[:, water_flat_indices].astype(np.float32))
    precip_water_hourly = xr.DataArray(precip_water_values,dims=("time_counter", "water_cell"),coords={
            "time_counter": precip_nemo_hourly["time_counter"].values,
            "water_cell": np.arange(n_water, dtype=np.int32),},name="precip",attrs=precip_nemo_hourly.attrs,)

    precip_water_3h = precip_water_hourly.resample(time_counter="3h",label="left",closed="left",origin="start_day",).mean()
    return precip_water_3h.load()

In [12]:
test_file = hrdps_files_by_year[2008][0]

test_3h = process_daily_file_to_3h_water(test_file)

print(test_3h)
print("Shape:", test_3h.shape)
print("Times:", test_3h.time_counter.values)

<xarray.DataArray 'precip' (time_counter: 8, water_cell: 81383)> Size: 3MB
array([[0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00],
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00],
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 1.7777133e-05,
        1.6897473e-05, 1.6039708e-05],
       ...,
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 1.3190848e-04,
        1.3063756e-04, 1.2938050e-04],
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 9.4836345e-05,
        9.3763250e-05, 9.2777547e-05],
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 1.2457016e-04,
        1.2062027e-04, 1.1674295e-04]], shape=(8, 81383), dtype=float32)
Coordinates:
  * time_counter  (time_counter) datetime64[ns] 64B 2008-01-01 ... 2008-01-01...
  * water_cell    (water_cell) int32 326kB 0 1 2 3 4 ... 81379 81380 81381 81382
Attributes:
    level:          

In [13]:
test_file = hrdps_files_by_year[2015][0]

test_3h = process_daily_file_to_3h_water(test_file)

print(test_3h)
print("Shape:", test_3h.shape)
print("Times:", test_3h.time_counter.values)

<xarray.DataArray 'precip' (time_counter: 8, water_cell: 81383)> Size: 3MB
array([[ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
         3.6844838e-08,  5.8053228e-08,  7.9986251e-08],
       [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
        -2.8653398e-08, -3.8412598e-08, -4.5285365e-08],
       [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
        -8.1914413e-09, -1.9640629e-08, -3.4700879e-08],
       ...,
       [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
         0.0000000e+00,  0.0000000e+00,  0.0000000e+00],
       [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
         8.4759422e-06,  8.4745525e-06,  8.4806579e-06],
       [ 0.0000000e+00,  0.0000000e+00,  0.0000000e+00, ...,
         1.5485381e-06,  1.6202326e-06,  1.6847169e-06]],
      shape=(8, 81383), dtype=float32)
Coordinates:
  * time_counter  (time_counter) datetime64[ns] 64B 2015-01-01 ... 2015-01-01...
  * water_cell    (water_cell) int32 326kB 0 1 2 3 4 ... 81379 81380 81

In [14]:
# Run only once for the remaining years
output_dir = "/ocean/dtaneja/MOAD/analysis-dishika/notebooks/data/hrdps_nemo_3h"
os.makedirs(output_dir, exist_ok=True)
remaining_years = [2007] + list(range(2013, 2022))
hrdps_nemo_processed_files = []

for year in remaining_years:
    files = hrdps_files_by_year[year]
    print(f"\nProcessing {year}: {len(files)} daily files")
    daily_3h_results = []

    for file_number, file in enumerate(files, start=1):
        daily_3h = process_daily_file_to_3h_water(file)
        daily_3h_results.append(daily_3h)

        if file_number == 1 or file_number % 25 == 0 or file_number == len(files):
            print(f"{year}: processed {file_number}/{len(files)} files")

    precip_year = xr.concat(daily_3h_results, dim="time_counter").sortby("time_counter")
    time_index = precip_year.get_index("time_counter")

    if time_index.has_duplicates:
        precip_year = precip_year.isel(time_counter=~time_index.duplicated())

    ds_year = precip_year.to_dataset(name="precip")
    ds_year = ds_year.assign_coords(nemo_j=("water_cell", nemo_j.astype(np.int32)), nemo_i=("water_cell", nemo_i.astype(np.int32)), nav_lat=("water_cell", nemo_water_lat.astype(np.float32)), nav_lon=("water_cell", nemo_water_lon.astype(np.float32)))
    ds_year.attrs["description"] = "Raw hourly HRDPS precip interpolated onto NEMO surface water cells, then resampled to three-hourly means."

    output_file = f"{output_dir}/HRDPS_NEMO_{year}_precip_3h.nc"
    encoding = {"precip": {"dtype": "float32", "zlib": True, "complevel": 4}}
    ds_year.to_netcdf(output_file, engine="netcdf4", encoding=encoding)
    hrdps_nemo_processed_files.append(output_file)

    print("Saved:", output_file)
    print("Shape:", ds_year["precip"].shape)
    print("Time range:", ds_year.time_counter.values[0], "to", ds_year.time_counter.values[-1])

    del daily_3h_results
    del precip_year
    del ds_year
    gc.collect()


Processing 2007: 363 daily files
2007: processed 1/363 files
2007: processed 25/363 files
2007: processed 50/363 files
2007: processed 75/363 files
2007: processed 100/363 files
2007: processed 125/363 files
2007: processed 150/363 files
2007: processed 175/363 files
2007: processed 200/363 files
2007: processed 225/363 files
2007: processed 250/363 files
2007: processed 275/363 files
2007: processed 300/363 files
2007: processed 325/363 files
2007: processed 350/363 files
2007: processed 363/363 files
Saved: /ocean/dtaneja/MOAD/analysis-dishika/notebooks/data/hrdps_nemo_3h/HRDPS_NEMO_2007_precip_3h.nc
Shape: (2904, 81383)
Time range: 2007-01-03T00:00:00.000000000 to 2007-12-31T21:00:00.000000000

Processing 2013: 365 daily files
2013: processed 1/365 files
2013: processed 25/365 files
2013: processed 50/365 files
2013: processed 75/365 files
2013: processed 100/365 files
2013: processed 125/365 files
2013: processed 150/365 files
2013: processed 175/365 files
2013: processed 200/365 